<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_05%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. CNN의 핵심 특징 강의에서 다룬 CNN(합성곱 신경망)이 기존의 완전 연결 신경망(DNN)보다 이미지 처리에 강력한 성능을 보이는 3가지 주요 특징이 있습니다. 가중치 공유(Weight Sharing) 외에 나머지 두 가지 특징을 서술하세요.

답변 작성란:

가중치 공유 (Weight Sharing)

자동 특징 추출 (Automatic Feature Exraction)

계층적 학습 (Hierarchical Learning)

문항 2. 전이 학습 (Transfer Learning) ImageNet과 같은 대규모 데이터셋으로 미리 학습된 모델(Pre-trained Model)의 가중치를 가져와서, 새로운 데이터셋이나 문제에 맞게 재학습시키는 기법을 무엇이라고 합니까? 또한 이 기법을 사용할 때의 장점을 한 가지 서술하세요.

답변 작성란:

용어: 파인튜닝

장점
- 데이터셋이 적어도 학습이 충분하다. 즉 높은 성능을 낼 수 있다.
- 학습 속도가 빠르다.
- 모델 수렴이 잘 된다.


문항 3. LeNet 모델 정의 - 합성곱 층 (Convolution Layers) LeNet-5 구조를 기반으로 한 LeNet 클래스의 __init__ 메서드입니다. 주석에 명시된 입출력 채널 수와 커널 크기에 맞춰 **합성곱 층(self.cn1, self.cn2)** 을 정의하는 코드를 작성하세요.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()

        # TODO: 첫 번째 합성곱 층 정의
        # 입력 채널: 3 (RGB), 출력 채널: 6, 커널 크기: 5x5
        self.cn1 = nn.Conv2d(3,6,5)

        # TODO: 두 번째 합성곱 층 정의
        # 입력 채널: 6, 출력 채널: 16, 커널 크기: 5x5
        self.cn2 = nn.Conv2d(6,16,5)

        # 완전 연결 층 정의 (이어서 작성할 문제)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.relu(self.cn1(x))
        x = F.max_pool2d(x, (2, 2))
        x = F.relu(self.cn2(x))
        x = F.max_pool2d(x, (2, 2))
        x = x.view(-1, self.flattened_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def flattened_features(self, x):
        size = x.size()[1:]
        num_feats = 1
        for s in size:
            num_feats *= s
        return num_feats

문항 4. LeNet 모델 정의 - 완전 연결 층 (Fully Connected Layers) 앞서 작성한 합성곱 층을 거친 후 출력되는 특징 맵을 1차원 벡터로 펼친 뒤, 분류를 수행하는 **완전 연결 층(self.fc1, self.fc2, self.fc3)** 을 정의하세요.

In [ ]:
class LeNet_Part2(nn.Module):
    def __init__(self):
        super(LeNet_Part2, self).__init__()
        self.cn1 = nn.Conv2d(3, 6, 5)
        self.cn2 = nn.Conv2d(6, 16, 5)

        # TODO: 완전 연결 층 정의
        # fc1: 입력 16*5*5 -> 출력 120
        self.fc1 = nn.Linear(16*5*5, 120)

        # fc2: 입력 120 -> 출력 84
        self.fc2 = nn.Linear(120, 84)

        # fc3: 입력 84 -> 출력 10 (CIFAR-10 클래스 개수)
        self.fc3 = nn.Linear(84, 10)

문항 5. 학습 루프 구현 (Training Loop) 모델 학습의 핵심 단계인 **역전파(Backpropagation)** 와 가중치 업데이트(Optimization) 과정을 구현하세요. (Optimizer는 optim, 손실 함수 결과는 loss 변수에 저장되어 있다고 가정합니다.)

In [ ]:
def train(net, trainloader, optim, epoch):
    loss_total = 0.0
    for i, data in enumerate(trainloader, 0):
        ip, ground_truth = data

        # TODO: 1. 변화도(Gradient) 초기화
        optim.zero_grad()

        # 순전파 및 손실 계산
        op = net(ip)
        loss = nn.CrossEntropyLoss()(op, ground_truth)

        # TODO: 2. 역전파 (Backpropagation) 수행
        loss.backward()

        # TODO: 3. 가중치 업데이트 (Optimization) 수행
        optim.step()

        loss_total += loss.item()
        if (i+1) % 1000 == 0:
            print(f'[Epoch: {epoch+1}, Mini-batches: {i+1}] loss: {loss_total/200:.3f}')
            loss_total = 0.0

문항 6. 모델 평가 및 예측 (Prediction) 테스트 데이터에 대해 모델의 성능을 평가할 때, 모델의 출력값(op) 중에서 가장 높은 확률을 가진 클래스의 인덱스를 찾아야 합니다. torch.max 함수를 사용하여 **예측 클래스(pred)** 를 구하는 코드를 작성하세요.

In [ ]:
import torch

In [ ]:
def test(net, testloader):
    success = 0
    counter = 0
    with torch.no_grad():
        for data in testloader:
            im, ground_truth = data
            op = net(im)

            # TODO: 예측값 구하기 (가장 높은 값을 가진 인덱스 추출)
            # 힌트: torch.max(tensor, dim) 사용
            _, pred = torch.max(op.data, 1)

            counter += ground_truth.size(0)
            success += (pred == ground_truth).sum().item()

    print(f'Accuracy: {100 * success / counter}%')